In [2]:
----- dim products
-----  update the category
--- if the category change will keep the old row with is_active=false

MERGE INTO dim_products as TARGET
using DELTA.`Files/silver/silver_products` as SOURCE
on TARGET.product_id_bk = SOURCE.product_id
and TARGET.is_active = true
when matched and TARGET.product_category_name <> source.product_category_name_english THEN
update set
target.is_active=false,
target.updated_at=current_timestamp() ;

---- if product category change insert the new raw with new category with is_active=true
INSERT INTO dim_products
SELECT 
    (
    SELECT COALESCE(MAX(product_key), 0) FROM dim_products) + row_number() OVER (ORDER BY SOURCE.product_id) AS product_key,
    Source.product_id as product_id_bk,
    Source.product_category_name_english,
    Source.product_name_length,
    Source.product_description_length,
    Source.product_photos_qty,
    Source.product_weight_g,
    Source.product_length_cm,
    Source.product_height_cm,
    Source.product_width_cm,
    current_timestamp() AS created_at,
    current_timestamp() AS updated_at,
    true AS is_active
FROM DELTA.`Files/silver/silver_products`AS SOURCE
WHERE NOT EXISTS ( 
    SELECT 1 FROM dim_products 
    WHERE product_id_bk = SOURCE.product_id 
      AND is_active = true 
      AND product_category_name = SOURCE.product_category_name_english
);



StatementMeta(, 2884bb91-ac9e-4699-b0e7-a1838ec10d73, 8, Finished, Available, Finished, True)

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 1 rows and 4 fields>

<Spark SQL result set with 0 rows and 0 fields>

In [6]:
------- dim selers

insert into dim_sellers
select( 
    select coalesce(max(seller_key),0) from dim_sellers) + row_number() over (ORDER BY seller_id) as seller_key ,
    seller_id ,
    seller_zip_code_prefix  , 
    seller_city  , 
    seller_state  , 
    geolocation_lat  , 
    geolocation_lng  ,
    current_timestamp() as created_at  
from  DELTA.`Files/silver/silver_sellers` ss
where not exists (
    select 1
    from dim_sellers  
    where dim_sellers.seller_id_bk = ss.seller_id
)

StatementMeta(, 804334bd-a23d-4245-b18a-679b15037889, 15, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [22]:
---- dim customers

insert into dim_customers
select
    ( select coalesce(max(customer_key),0) from dim_customers )+row_number() over (order by customer_id) as customer_key,
    customer_id , 
    customer_unique_id , 
    customer_zip_code_prefix,
    customer_city , 
    customer_state ,
    geolocation_lat,
    geolocation_lng,
    current_timestamp() as created_at
from delta.`Files/silver/silver_customers` as sc
where not exists (
    select 1
    from dim_customers 
    where sc.customer_id = dim_customers.customer_id_bk
)


StatementMeta(, 804334bd-a23d-4245-b18a-679b15037889, 35, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [8]:
----- dim date

INSERT INTO dim_date
WITH SimpleNumbers AS (
    SELECT id AS days_offset FROM range(0, 10001)
),
DateBase AS (
    SELECT DATE_ADD(CAST('2010-01-01' AS DATE), CAST(days_offset AS INT)) AS date_value
    FROM SimpleNumbers
    WHERE DATE_ADD(CAST('2010-01-01' AS DATE), CAST(days_offset AS INT)) <= CAST('2030-12-31' AS DATE)
)
SELECT 
    CAST(DATE_FORMAT(date_value, 'yyyyMMdd') AS INT) AS date_key,
    date_value AS full_date,
    YEAR(date_value) AS year,
    CONCAT('Q', QUARTER(date_value)) AS quarter,
    MONTH(date_value) AS month,
    DATE_FORMAT(date_value, 'MMMM') AS month_name,
    WEEKOFYEAR(date_value) AS week_of_year,
    DAY(date_value) AS day_of_month,
    DATE_FORMAT(date_value, 'EEEE') AS day_name,
    CASE WHEN DAYOFWEEK(date_value) IN (1, 7) THEN 1 ELSE 0 END AS is_weekend
FROM DateBase;

StatementMeta(, 2acb6de8-1565-49ce-8d84-43a74678ebc8, 13, Finished, Available, Finished, True)

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

In [7]:
---- dim time

INSERT INTO dim_time
WITH MinuteSeries AS (
    SELECT id AS mins FROM range(0, 1440)
)
SELECT 
    -- الـ Key يفضل يفضل يفضل يكون بصيغة HHMMSS (مثلاً 143000) لسهولة الترتيب
    (CAST(mins / 60 AS INT) * 10000) + ((mins % 60) * 100) AS time_key,
    
    -- التعديل الجوهري: إضافة :00 في الآخر لتطابق صيغة HH:mm:ss
    PRINTF('%02d:%02d:00', CAST(mins / 60 AS INT), mins % 60) AS time_value,
    
    CAST(mins / 60 AS INT) AS hour,
    mins % 60 AS minute,
    0 AS second, -- الثواني دايماً أصفار لأن الحسبة بالدقائق
    CASE WHEN CAST(mins / 60 AS INT) >= 12 THEN 'PM' ELSE 'AM' END AS period
FROM MinuteSeries;

StatementMeta(, 2ae745a7-cdae-4cd7-bb0e-9147318997ab, 11, Finished, Available, Finished, True)

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

In [34]:
----- fact orders

insert into table fact_orders
select
    (select coalesce(max(order_key),0)from fact_orders) + row_number() over (order by so.order_id) as order_key ,
    so.order_id ,
    dc.customer_key,
    so.order_status,
    so.order_purchase_timestamp,
    dim_date_purchase.date_key,
    dim_time_purchase.time_key,
    so.order_approved_at,
    dim_date_approved.date_key,
    dim_time_approved.time_key,
    so.approved_performance ,
    dim_date_delivere_carrier.date_key,
    dim_time_delivere_carrier.time_key,
    dim_date_delivere_customer.date_key,
    dim_time_delivere_customer.time_key,
    so.order_estimated_delivery_date,
    so.delivery_performance,
    so.total_process,
    so.red_flag,
    sor.review_score,
    sor.review_comment_title,
    sor.review_comment_message,
    current_timestamp() as created_at 


from delta.`Files/silver/silver_orders` as so

---- join with fact to prevent duplicates
left join fact_orders fo on so.order_id = fo.order_id_bk

---- customers
left join dim_customers dc on dc.customer_id_bk = so.customer_id

---- purchase
left join dim_date dim_date_purchase on dim_date_purchase.full_date = so.order_purshase_date
left join dim_time dim_time_purchase on dim_time_purchase.time_value = so.order_purshase_time                         

---- approved
left join dim_date dim_date_approved on dim_date_approved.full_date = so.order_approved_date
left join dim_time dim_time_approved on dim_time_approved.time_value = so.order_approved_time

---- delivered customer
left join dim_date dim_date_delivere_customer on dim_date_delivere_customer.full_date = so.order_delivered_customer_date
left join dim_time dim_time_delivere_customer on dim_time_delivere_customer.time_value = so.order_delivered_customer_time

---- delivered carrier
left join dim_date dim_date_delivere_carrier on dim_date_delivere_carrier.full_date = so.order_delivered_carrier_date
left join dim_time dim_time_delivere_carrier on dim_time_delivere_carrier.time_value = so.order_delivered_carrier_time

---- reviews 
left join delta.`Files/silver/silver_order_reviews` as sor on sor.order_id=so.order_id

WHERE fo.order_id_bk is NULL




StatementMeta(, 804334bd-a23d-4245-b18a-679b15037889, 50, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [5]:
----  fact order items

insert into fact_order_items
select
    (select coalesce(max(order_item_key),0) from fact_order_items) + row_number() over(order by soi.product_id,soi.order_id) as order_item_key,
    
    fo.order_key,
    dp.product_key,
    ds.seller_key,
    
    soi.Total_QTY, 
    soi.Unit_price,
    soi.Unit_freight,
    soi.Total_product_price,
    soi.Total_freight,
    soi.Total_order_value,
    soi.shipping_limit_date,

    current_timestamp as created_at
from delta.`Files/silver/silver_order_items` as soi
--- fact order 
left join fact_orders fo on fo.order_id_bk = soi.order_id
--- product 
left join dim_products dp on dp.product_id_bk = soi.product_id
--- seller
left join dim_sellers ds on ds.seller_id_bk = soi.seller_id
where not exists (
    select 1
    from fact_order_items foi
    where foi.order_id_fk = fo.order_key
    and   foi.product_id_fk = dp.product_key
    and   foi.seller_id_fk = ds.seller_key
)


StatementMeta(, 7fd32f8c-2ef3-400e-88e8-9f0f886f5152, 6, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [4]:
----- fact order payments 

insert into fact_order_payments
select 
    (select coalesce(max(order_payment_key),0) from fact_order_payments) + row_number() over (order by sop.order_id ) as order_payment_key,
    fo.order_key , 
    sop.payment_sequential,
    sop.payment_type,
    sop.payment_installments,
    sop.payment_value,

    current_timestamp() as created_at

from delta.`Files/silver/silver_order_payments` as sop

--- fact order 
left join fact_orders fo on fo.order_id_bk = sop.order_id
where not exists(

    select 1
    from fact_order_payments fop
    where fop.order_id_fk = fo.order_key
    and   fop.payment_sequential = sop.payment_sequential
)

StatementMeta(, c18965bd-e70a-4571-9def-83c0ad54c64e, 5, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>